In [1]:
import pandas as pd
df = pd.read_csv("messy_sales.csv")
print(df.shape)
df.head(10)

(300, 6)


,order_id,date,product,price,qty,zip
0,1254,04/06/2026,mug,11.36,4,60614
1,1057,2026-05-30,charger,14.79,3,30303
2,1150,05/06/2026,mug,5.50,1,10001
3,1066,05/30/2026,notebook,NaN,1,98101
4,1114,04/06/2026,webcam,45.59,4,90405
5,1280,04/22/2026,pen set,NaN,4,30303
6,1204,04/18/2026,charger,8.41,2,2134
7,1249,2026-05-15,keyboard,75.04,4,90405
8,1037,2026-05-19,mug,9.31,2,60614
9,1177,2026-05-13,mug,10.11,3,90210


In [2]:
df.dtypes

order_id      int64
date            str
product         str
price       float64
qty           int64
zip           int64
dtype: object

zip codes lost their leading zeros because they’re stored as numbers.
dates are stored as text in two mixed formats.
some prices are blank, and some quantities are negative.

In [3]:
df.isna().sum()

order_id     0
date         0
product      0
price       12
qty          0
zip          0
dtype: int64

In [4]:
fill_value = df["price"].median()
print("Fill value:", fill_value)
df["price"] = df["price"].fillna(fill_value)
df.isna().sum()

Fill value: 37.53


order_id    0
date        0
product     0
price       0
qty         0
zip         0
dtype: int64

In [5]:
print("Duplicates:", df.duplicated().sum())
df = df.drop_duplicates()
print(df.shape)

Duplicates: 8
(292, 6)


In [6]:
df["zip"] = df["zip"].astype(str).str.zfill(5)
df["zip"].head(10)

0    60614
1    30303
2    10001
3    98101
4    90405
5    30303
6    02134
7    90405
8    60614
9    90210
Name: zip, dtype: str

In [7]:
df["date"] = pd.to_datetime(df["date"], format="mixed")
df.dtypes

order_id             int64
date        datetime64[us]
product                str
price              float64
qty                  int64
zip                    str
dtype: object

In [8]:
df[df["qty"] < 0]

,order_id,date,product,price,qty,zip
202,1140,2026-04-18,webcam,38.69,-5,98101
262,1233,2026-05-09,desk lamp,22.64,-4,10001
297,1025,2026-04-27,keyboard,47.30,-2,02116


In [9]:
df["is_return"] = df["qty"] < 0
df[df["is_return"]]

,order_id,date,product,price,qty,zip,is_return
202,1140,2026-04-18,webcam,38.69,-5,98101,True
262,1233,2026-05-09,desk lamp,22.64,-4,10001,True
297,1025,2026-04-27,keyboard,47.30,-2,02116,True


In [10]:
df.to_csv("sales_clean.csv", index=False)
df.shape

(292, 7)

Cleaning Log

1. Loaded 300 rows and 6 columns from messy_sales.csv.
2. Missing prices: 12 prices were blank. I filled them with the median price, 37.53, because the median is not pulled around by a few very expensive items the way the mean is.
3. Duplicates: I found and removed 8 fully repeated rows, leaving 292 rows.
4. Zip codes: zips were stored as numbers, so leading zeros were lost (2134 instead of 02134). I converted them to text and padded them to 5 characters.
5. Dates: the date column was text in two formats. I converted it to real datetimes so it can be sorted and compared.
6. Negative quantities: 3 rows had negative quantities (-5, -4, and -2). I treated them as product returns, kept them, and added an is_return column marked True for these rows. This is why the final shape is 292 rows by 7 columns instead of 292 by 6.
7. Saved the cleaned data to sales_clean.csv without the index column.

If I had filled the prices with the mean, or deleted the returns, total sales would look higher than they really were, and a manager could overorder stock or overestimate revenue.

https://github.com/harveyvasquez/cs82a-portfolio